# Pareto-HAR v2 — Channel-Independent Spectro-Temporal Masked Pretraining with Subject-Adversarial Fine-tuning

**A reproducible, journal-grade study of subject-independent Human Activity Recognition.**

---
### What changed vs. v1 (and why)
| v1 problem | v2 fix |
|---|---|
| PAMAP2 (18 ch) excluded from SSL — channel-count mismatch | **Channel-Independent Tokenizer (CIT)**: one backbone for any channel count → unified cross-dataset pretraining |
| SSL pool used *all* WISDM windows incl. test subjects (**leakage**) | SSL uses **training-subject windows only**, re-built per LOSO fold |
| "MAE" kept all tokens (denoising AE) | **Asymmetric MAE**: encoder sees only visible tokens (faster, stronger reps) |
| Time-only reconstruction | **Spectro-temporal** loss (time + DFT magnitude) — captures gait rhythmicity |
| train 100 / test 86.6 (subject memorization) | **Subject-Adversarial Fine-tuning (SAFT)** via gradient reversal → subject-invariant features |
| mixup crash (`int64` one-hot) | fixed; PAMAP2 & ExtraSensory now run |
| single 80/20 split | **Leave-One-Subject-Out (LOSO)** cross-validation + per-fold stats |
| accuracy / weighted-F1 only | **full battery**: macro-F1, balanced acc, κ, MCC, AUC, ECE, per-class |
| no significance testing | Wilcoxon, paired-t, Cohen's d, McNemar, Friedman+Nemenyi, bootstrap CIs |
| no honest baselines | DeepConvLSTM, CNN-LSTM, plain Transformer, contrastive (TS-TCC-style) on identical folds |
| no resume — a disconnect lost everything | **fold-level checkpointing + auto-resume** (cached predictions/metrics + backbone weights) |
| fp32, default Keras loop | **mixed-precision (fp16 tensor cores) + XLA + static-shape `tf.data`** for high GPU utilisation |
| final-epoch model, no class balancing | **val-based early stopping / LR-decay, gradient clipping, class-weighted loss** |

### Proposed method = three contributions (each maps to a real gap)
1. **CIT** — channel-as-token + learned channel/positional embeddings → heterogeneous sensors share one encoder.
2. **ST-MAE** — asymmetric masked autoencoder reconstructing both time-domain values and spectral magnitude.
3. **SAFT** — gradient-reversal subject classifier removes identity nuisance, closing the LOSO gap.

> **Honest framing.** Each ingredient has lineage (CIT≈PatchTST, ST-MAE≈CRT/MAE, SAFT≈DANN). The contribution is their *integration targeting the subject-generalization gap*, under a rigorous LOSO protocol with full statistics. On honest LOSO protocols, published HAR sits ~85–92% (WISDM) and ~90–94% (PAMAP2); the inflated 98–99% figures use leaky window-level random splits.


## 1 — Runtime & GPU check
Runs on a **Lightning AI Studio** (or any Jupyter environment). Attach a GPU to the Studio for fast training; the code also runs on CPU, slowly.

In [ ]:
import subprocess, sys
print(sys.version)
try:
    print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader']).decode())
except Exception:
    print('No GPU detected — runs on CPU but slowly. Attach a GPU to your Lightning Studio.')


## 2 — Install dependencies (GPU-safe)
Lightning AI images are **PyTorch-first**: a plain `tensorflow` install can't find cuDNN/cuBLAS there and **silently runs on CPU** even with a GPU attached. This cell checks TF's GPU visibility in a *subprocess* (so this kernel is never poisoned by a failed CUDA init), installs `tensorflow[and-cuda]` if needed, restarts the kernel once if a CPU-only TF is already loaded, and **refuses to proceed on CPU when a GPU exists**.

In [ ]:
import importlib, subprocess, sys, os

def _run(*cmd):
    try:
        return subprocess.run(list(cmd), capture_output=True, text=True)
    except FileNotFoundError:
        class _R: returncode=127; stdout=''; stderr=f'{cmd[0]}: not found'
        return _R()

def _pip(*args):
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*args])

HAS_NVIDIA = _run('nvidia-smi','-L').returncode == 0

for pkg, mod in [('statsmodels',None), ('seaborn',None), ('scikit-learn','sklearn')]:
    try: importlib.import_module(mod or pkg)
    except ImportError: print('installing', pkg, '...'); _pip(pkg)

def _tf_gpu_count():
    '''TF GPU visibility checked in a FRESH process — never poisons this kernel.'''
    code=("import os;os.environ['TF_CPP_MIN_LOG_LEVEL']='3';"
          "import tensorflow as tf;print('NGPU='+str(len(tf.config.list_physical_devices('GPU'))))")
    r=_run(sys.executable,'-c',code)
    for line in r.stdout.splitlines():
        if line.startswith('NGPU='): return int(line.split('=')[1]), r.stderr
    return -1, (r.stderr or r.stdout)   # TF missing or crashed on import

if HAS_NVIDIA:
    n, err = _tf_gpu_count()
    if n <= 0:
        print('NVIDIA GPU present but TensorFlow cannot see it.')
        print('Installing tensorflow[and-cuda] (bundles matching cuDNN/cuBLAS via pip, ~2 min) ...')
        _pip('-U','tensorflow[and-cuda]')
        n, err = _tf_gpu_count()
        if n <= 0:
            print('STILL no GPU visible to TensorFlow. Loader errors from the test process:')
            print((err or '')[-2000:])
            print(_run('nvidia-smi').stdout[:600])
            raise SystemExit('Fix the TF/CUDA install before training — refusing to run on CPU by accident.')
    # If an earlier cell already imported a CPU-only TF, its failed CUDA init is cached
    # for the process lifetime -> a kernel restart is REQUIRED (re-importing is not enough).
    if 'tensorflow' in sys.modules:
        import tensorflow as tf
        if not tf.config.list_physical_devices('GPU'):
            print('\n=== A CPU-only TensorFlow is already loaded in this kernel.            ===')
            print('=== RESTARTING the kernel so the GPU build loads. Then: Run All again. ===')
            print('=== (The resume cache skips all completed folds automatically.)        ===')
            try:
                import IPython; IPython.Application.instance().kernel.do_shutdown(True)
            except Exception:
                raise SystemExit('Restart the kernel manually (Kernel -> Restart Kernel), then Run All.')
else:
    try: importlib.import_module('tensorflow')
    except ImportError: print('installing tensorflow (CPU) ...'); _pip('tensorflow')

import tensorflow as tf, numpy as np, pandas as pd, sklearn, scipy, statsmodels
print('TF', tf.__version__, '| sklearn', sklearn.__version__, '| scipy', scipy.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus if gpus else 'none — running on CPU')
if HAS_NVIDIA and not gpus:
    raise SystemExit('A GPU exists but TF cannot see it — refusing to train on CPU. See messages above.')


## 3 — Paths, reproducibility & global config
No cloud mount needed. A **Lightning AI Studio's filesystem is persistent** — everything written below (data, checkpoints, results, the resume cache) survives Studio stops/restarts and GPU swaps. Paths default to a folder next to the notebook; override `OUTPUT_ROOT` to point anywhere persistent.

In [ ]:
import os, random
from datetime import datetime

# Persistent project root. On a Lightning Studio the whole filesystem persists,
# so a local folder is durable. Auto-detect the Studio workspace if present.
def _default_root():
    for p in ['/teamspace/studios/this_studio', os.path.expanduser('~')]:
        if os.path.isdir(p): return os.path.join(p, 'Pareto_HAR_v2')
    return os.path.join(os.getcwd(), 'Pareto_HAR_v2')

OUTPUT_ROOT   = os.environ.get('PARETO_ROOT', _default_root())
DATA_DIR      = f'{OUTPUT_ROOT}/data'
CKPT_DIR      = f'{OUTPUT_ROOT}/checkpoints'
RESULTS_DIR   = f'{OUTPUT_ROOT}/results'
PLOTS_DIR     = f'{OUTPUT_ROOT}/plots'
FOLDCACHE_DIR = f'{RESULTS_DIR}/foldcache'   # per-fold predictions+metrics (enables resume)
JOURNAL_PATH  = f'{RESULTS_DIR}/metrics_journal.json'
for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR, PLOTS_DIR, FOLDCACHE_DIR]:
    os.makedirs(d, exist_ok=True)

# A FIXED run id keeps checkpoints/cache stable across disconnects so the study RESUMES.
# (Override by setting RUN_ID yourself; use a fresh id only when you want a clean slate.)
RUN_ID = os.environ.get('PARETO_RUN_ID', 'main')

# -------------------- GLOBAL CONFIG (tune for your compute budget) --------------------
SEED              = 42
WINDOW_SIZE       = 128
STRIDE            = 64
PATCH             = 16          # CIT patch length -> N = WINDOW_SIZE//PATCH tokens per channel

# model size (all dims multiples of 8 -> fp16 tensor-core friendly)
D_MODEL, N_HEADS, N_ENC, N_DEC, FF_DIM, DROPOUT = 128, 8, 4, 2, 256, 0.1

# pretraining (batches sized for ~16 GB GPU under mixed precision; raise on bigger cards, lower if you OOM)
MASK_RATIO, MAE_EPOCHS, MAE_BATCH, MAE_LR = 0.60, 40, 512, 3e-4
LAMBDA_T, LAMBDA_F = 1.0, 0.5   # time vs frequency reconstruction weights

# fine-tuning
FT_EPOCHS, FT_BATCH, FT_LR = 60, 256, 2e-4
GRAD_CLIPNORM = 1.0
SAFT_GAMMA, SAFT_MAX_ALPHA = 10.0, 1.0   # DANN ramp; set SAFT_MAX_ALPHA=0 to disable adversary

# ---- performance switches (the "use the GPU hard" knobs) ----
USE_MIXED_PRECISION = True     # bf16 on Ampere+ (A100/H100), fp16 elsewhere; engages only when a GPU is present
USE_XLA             = True     # XLA JIT compile of train steps; big speedup with static per-fold shapes
USE_CLASS_WEIGHTS   = True     # balance loss for skewed datasets (e.g. ExtraSensory)
STEPS_PER_EXEC      = 32       # fused train steps per tf.function dispatch — hides Python overhead;
                               # for small models this is the biggest GPU-utilisation win on A100-class cards
AUTO_SCALE_BATCH    = True     # scale batches (and sqrt-scale LRs) up on large-VRAM GPUs (A100 40/80GB)

# ---- robustness / resume ----
RESUME            = True       # skip folds already cached -> safe to re-run after a disconnect
SAVE_CHECKPOINTS  = True       # save the shared backbone (tokenizer+encoder) weights per completed fold

# evaluation protocol
LOSO_MAX_FOLDS    = 5          # cap held-out subjects (None = all). 5 keeps a single-GPU run reasonable.
QUICK_SMOKE       = False      # True: subsample everything to validate the pipeline in minutes
DATASETS_TO_RUN   = ['WISDM', 'PAMAP2', 'ExtraSensory']
RUN_BASELINES     = True
RUN_ABLATIONS     = True

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# -------------------- hardware setup: memory growth + mixed precision + XLA --------------------
import subprocess
from tensorflow.keras import mixed_precision
GPUS = tf.config.list_physical_devices('GPU')
for g in GPUS:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass

MIXED = USE_MIXED_PRECISION and len(GPUS) > 0
if MIXED:
    cc = (0, 0)
    try: cc = tuple(tf.config.experimental.get_device_details(GPUS[0]).get('compute_capability', (0, 0)))
    except Exception: pass
    # Ampere+ (A100/H100, cc>=8.0): bfloat16 — fp32-range exponent, NO loss scaling needed, more stable.
    # Older cards (T4/V100): float16 with loss scaling (handled automatically by make_optimizer).
    POLICY = 'mixed_bfloat16' if cc >= (8, 0) else 'mixed_float16'
    mixed_precision.set_global_policy(POLICY)
else:
    mixed_precision.set_global_policy('float32')  # CPU / mixed precision off

if USE_XLA and len(GPUS) > 0:
    try: tf.config.optimizer.set_jit(True)
    except Exception: pass

# ---- large-GPU auto scaling: bigger batches (+ sqrt-scaled LRs) to fill an A100 ----
if AUTO_SCALE_BATCH and GPUS:
    try:
        vram = int(subprocess.check_output(['nvidia-smi','--query-gpu=memory.total',
                  '--format=csv,noheader,nounits']).decode().split()[0])
        mult = 4 if vram >= 70000 else (2 if vram >= 30000 else 1)
        if mult > 1:
            MAE_BATCH *= mult; FT_BATCH *= mult
            MAE_LR *= mult ** 0.5; FT_LR *= mult ** 0.5   # sqrt LR scaling for larger batches
            print(f'AUTO_SCALE_BATCH: {vram//1024} GB GPU -> x{mult} batches '
                  f'(MAE_BATCH={MAE_BATCH}, FT_BATCH={FT_BATCH}), LRs x{mult**0.5:.2f}')
    except Exception: pass

print(f'Run ID {RUN_ID} | results -> {RESULTS_DIR}')
print(f'GPUs={len(GPUS)} | precision={mixed_precision.global_policy().name} | XLA={USE_XLA and len(GPUS)>0} '
      f'| steps_per_execution={STEPS_PER_EXEC}')
print(f'RESUME={RESUME} | checkpoints={SAVE_CHECKPOINTS} | MAE_BATCH={MAE_BATCH} FT_BATCH={FT_BATCH}')


## 4 — Persistent metrics journal (survives disconnects)

In [ ]:
import json
class MetricsJournal:
    def __init__(self, path):
        self.path = path
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if not os.path.exists(path):
            json.dump([], open(path, 'w'))
        print(f'Journal at {path}')
    def _load(self):
        return json.load(open(self.path)) if os.path.exists(self.path) else []
    def log(self, phase, dataset, fold, epoch, metrics):
        rec = self._load()
        rec.append({'run_id': RUN_ID, 'ts': datetime.now().isoformat(),
                    'phase': phase, 'dataset': dataset, 'fold': fold,
                    'epoch': epoch, **metrics})
        json.dump(rec, open(self.path, 'w'), indent=2)
    def df(self):
        return pd.DataFrame(self._load())
journal = MetricsJournal(JOURNAL_PATH)


## 5 — Data: download & load (WISDM, PAMAP2, ExtraSensory)
Loaders ported from v1. WISDM = 3-ch accelerometer; PAMAP2 = 18-ch (3 IMUs × acc+gyro); ExtraSensory = 3-ch accelerometer means.

In [ ]:
import tarfile, requests
WISDM_PATH = f'{DATA_DIR}/wisdm.txt'
def extract_wisdm_from_tar(tar_path, out):
    with tarfile.open(tar_path, 'r:gz') as tar:
        for m in tar.getmembers():
            if m.name.endswith('WISDM_ar_v1.1_raw.txt'):
                open(out,'wb').write(tar.extractfile(m).read()); return True
    return False
if not os.path.exists(WISDM_PATH):
    print('Downloading WISDM...')
    url='https://www.cis.fordham.edu/wisdm/includes/datasets/latest/WISDM_ar_latest.tar.gz'
    tp=f'{DATA_DIR}/wisdm.tar.gz'
    r=requests.get(url,timeout=60,stream=True); r.raise_for_status()
    with open(tp,'wb') as f:
        for c in r.iter_content(8192): f.write(c)
    assert extract_wisdm_from_tar(tp, WISDM_PATH), 'raw file not found'
    os.remove(tp)
def load_wisdm(path):
    rows=[]
    for line in open(path):
        line=line.strip().rstrip(';').strip()
        if not line: continue
        p=line.split(',')
        if len(p)!=6: continue
        try: rows.append([p[0].strip(),p[1].strip(),int(p[2]),float(p[3]),float(p[4]),float(p[5])])
        except ValueError: continue
    return pd.DataFrame(rows, columns=['user','activity','timestamp','x','y','z'])
wisdm_df = load_wisdm(WISDM_PATH)
print(f'WISDM: {len(wisdm_df):,} rows | {wisdm_df.activity.nunique()} acts | {wisdm_df.user.nunique()} users')


In [ ]:
# PAMAP2 (18 channels)
import io, zipfile
PAMAP2_DIR=f'{DATA_DIR}/pamap2'; PAMAP2_COMBINED=f'{PAMAP2_DIR}/pamap2_combined.csv'
os.makedirs(PAMAP2_DIR, exist_ok=True)
PAMAP2_LABEL_MAP={1:'lying',2:'sitting',3:'standing',4:'walking',5:'running',6:'cycling',
                  7:'nordic_walking',12:'ascending_stairs',13:'descending_stairs',
                  16:'vacuum_cleaning',17:'ironing',24:'rope_jumping'}
if os.path.exists(PAMAP2_COMBINED):
    pamap2_df=pd.read_csv(PAMAP2_COMBINED)
else:
    print('Downloading PAMAP2...')
    url='https://archive.ics.uci.edu/static/public/231/pamap2+physical+activity+monitoring.zip'
    r=requests.get(url,timeout=120); r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as oz:
        inner=next(n for n in oz.namelist() if n.lower().endswith('.zip'))
        with oz.open(inner) as izf, zipfile.ZipFile(io.BytesIO(izf.read())) as iz:
            dat=[n for n in iz.namelist() if n.endswith('.dat') and 'Protocol/' in n]
            dfs=[]
            for dn in dat:
                sid=int(dn.split('subject')[-1].split('.')[0])
                tmp=pd.read_csv(iz.open(dn), sep=' ', header=None)
                tmp.columns=[f'c{i}' for i in range(tmp.shape[1])]
                dfs.append(pd.DataFrame({'timestamp':tmp.c0,'activityID':tmp.c1,'subjectID':sid,
                    'hand_acc_x':tmp.c4,'hand_acc_y':tmp.c5,'hand_acc_z':tmp.c6,
                    'hand_gyro_x':tmp.c7,'hand_gyro_y':tmp.c8,'hand_gyro_z':tmp.c9,
                    'chest_acc_x':tmp.c10,'chest_acc_y':tmp.c11,'chest_acc_z':tmp.c12,
                    'chest_gyro_x':tmp.c13,'chest_gyro_y':tmp.c14,'chest_gyro_z':tmp.c15,
                    'ankle_acc_x':tmp.c16,'ankle_acc_y':tmp.c17,'ankle_acc_z':tmp.c18,
                    'ankle_gyro_x':tmp.c19,'ankle_gyro_y':tmp.c20,'ankle_gyro_z':tmp.c21}))
            pamap2_df=pd.concat(dfs, ignore_index=True); pamap2_df.to_csv(PAMAP2_COMBINED,index=False)
pamap2_df=pamap2_df[pamap2_df.activityID>0].dropna().copy()
pamap2_df['activity']=pamap2_df.activityID.map(PAMAP2_LABEL_MAP).fillna('other')
PAMAP2_FEAT_COLS=['hand_acc_x','hand_acc_y','hand_acc_z','hand_gyro_x','hand_gyro_y','hand_gyro_z',
 'chest_acc_x','chest_acc_y','chest_acc_z','chest_gyro_x','chest_gyro_y','chest_gyro_z',
 'ankle_acc_x','ankle_acc_y','ankle_acc_z','ankle_gyro_x','ankle_gyro_y','ankle_gyro_z']
print(f'PAMAP2: {len(pamap2_df):,} rows | {pamap2_df.activity.nunique()} acts | {pamap2_df.subjectID.nunique()} subjects')


In [ ]:
# ExtraSensory (3-ch acc means) — ported; falls back to synthetic only if download blocked
import gzip, re
ES_DIR=f'{DATA_DIR}/extrasensory'; ES_CSV=f'{ES_DIR}/extrasensory_combined.csv'
ES_ZIP=f'{ES_DIR}/ExtraSensory.per_uuid_features_labels.zip'; os.makedirs(ES_DIR, exist_ok=True)
if os.path.exists(ES_CSV):
    es_df=pd.read_csv(ES_CSV)
else:
    try:
        if not os.path.exists(ES_ZIP):
            url='http://extrasensory.ucsd.edu/data/primary_data_files/ExtraSensory.per_uuid_features_labels.zip'
            print('Downloading ExtraSensory (~215MB)...')
            r=requests.get(url,stream=True,timeout=180); r.raise_for_status()
            with open(ES_ZIP,'wb') as f:
                for c in r.iter_content(8192): f.write(c)
        with zipfile.ZipFile(ES_ZIP) as zf:
            files=[n for n in zf.namelist() if n.endswith('.csv.gz')]
            cols=pd.read_csv(gzip.GzipFile(fileobj=zf.open(files[0]))).columns.tolist()
            ax=next(c for c in cols if re.search(r'raw_acc.*[xX].*mean',c,re.I))
            ay=next(c for c in cols if re.search(r'raw_acc.*[yY].*mean',c,re.I))
            az=next(c for c in cols if re.search(r'raw_acc.*[zZ].*mean',c,re.I))
            lcols=[c for c in cols if c.startswith('label:')]
            rows=[]
            for fn in files:
                uid=os.path.splitext(os.path.splitext(fn)[0])[0]
                du=pd.read_csv(gzip.GzipFile(fileobj=zf.open(fn)))
                acc=du[[ax,ay,az]].fillna(0).values; lab=du[lcols].fillna(0).values
                for i in range(len(du)):
                    if lab[i].max()==0: continue
                    rows.append({'user':uid,'acc_x':acc[i,0],'acc_y':acc[i,1],'acc_z':acc[i,2],
                                 'activity':lcols[int(np.argmax(lab[i]))].replace('label:','')})
            es_df=pd.DataFrame(rows); assert len(es_df)>=10000
            es_df.to_csv(ES_CSV,index=False)
    except Exception as e:
        print('ExtraSensory download failed:', e, '-> synthetic dev fallback')
        rng=np.random.default_rng(SEED)
        acts=['FIX_walking','FIX_running','SITTING','LYING_DOWN','OR_standing',
              'STAIRS_-_GOING_UP','STAIRS_-_GOING_DOWN','BICYCLING']
        rows=[{'user':u,'activity':(a:=rng.choice(acts)),
               'acc_x':rng.normal(0,0.5 if 'walk' in a or 'run' in a else 0.1),
               'acc_y':rng.normal(0,0.5),'acc_z':rng.normal(9.8,0.5)}
              for u in range(1,61) for _ in range(5000)]
        es_df=pd.DataFrame(rows); es_df.to_csv(ES_CSV,index=False)
print(f'ExtraSensory: {len(es_df):,} rows | {es_df.activity.nunique()} acts | {es_df.user.nunique()} users')


## 6 — Preprocessing → windows, labels, **per-subject grouping** (for LOSO)
Normalisation is fit **inside each LOSO fold** later (no test leakage); here we only window.

In [ ]:
from sklearn.preprocessing import LabelEncoder
def sliding_window(data, labels, groups, win=WINDOW_SIZE, stride=STRIDE):
    X,y,g=[],[],[]
    for s in range(0, len(data)-win+1, stride):
        seg=labels[s:s+win]; vals,cnts=np.unique(seg,return_counts=True)
        # require a single dominant group in the window (clean LOSO grouping)
        gseg=groups[s:s+win]
        if len(np.unique(gseg))>1: continue
        X.append(data[s:s+win]); y.append(vals[np.argmax(cnts)]); g.append(gseg[0])
    return np.asarray(X,np.float32), np.asarray(y), np.asarray(g)

def build_windows(df, feat, label_col, user_col):
    le=LabelEncoder(); labels=le.fit_transform(df[label_col].values)
    feats=df[feat].values.astype(np.float32)
    groups=df[user_col].values
    X,y,g=sliding_window(feats, labels, groups)
    return X,y,g,le

DATA={}
X,y,g,le=build_windows(wisdm_df,['x','y','z'],'activity','user');           DATA['WISDM']=(X,y,g,le)
X,y,g,le=build_windows(pamap2_df,PAMAP2_FEAT_COLS,'activity','subjectID');   DATA['PAMAP2']=(X,y,g,le)
X,y,g,le=build_windows(es_df,['acc_x','acc_y','acc_z'],'activity','user');   DATA['ExtraSensory']=(X,y,g,le)
for k,(X,y,g,le) in DATA.items():
    print(f'{k:13s} X={X.shape} classes={len(le.classes_)} subjects={len(np.unique(g))}')

if QUICK_SMOKE:
    for k,(X,y,g,le) in DATA.items():
        idx=np.random.default_rng(0).choice(len(X), min(2000,len(X)), replace=False)
        DATA[k]=(X[idx],y[idx],g[idx],le)
    print('QUICK_SMOKE on — subsampled to <=2000 windows/dataset')


## 7 — Proposed architecture
**(a) Gradient-Reversal Layer (GRL)** for SAFT, **(b) Channel-Independent Tokenizer (CIT)**, **(c) shared Transformer encoder**, **(d) asymmetric Spectro-Temporal MAE**, **(e) classifier with attention pooling + subject-adversarial head.**

Mathematical formulation is in the companion `METHODS.md`. Key losses:
- Pretraining: $\mathcal{L}_{rec}=\frac{1}{|\mathcal{M}|}\sum_{(c,n)\in\mathcal{M}}\big[\lambda_t\lVert x_{c,n}-\hat x_{c,n}\rVert_2^2+\lambda_f\big\lVert |\mathcal{F}(x_{c,n})|-|\mathcal{F}(\hat x_{c,n})|\big\rVert_2^2\big]$
- Fine-tuning: $\min_{\theta_f,\theta_y}\max_{\theta_d}\;\mathcal{L}_y-\alpha\,\mathcal{L}_d$, GRL implements the inner max; $\alpha_p=\frac{2}{1+e^{-\gamma p}}-1$.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

@tf.custom_gradient
def _grad_reverse(x, alpha):
    def grad(dy): return -alpha*dy, None
    return tf.identity(x), grad

@keras.utils.register_keras_serializable(package='HAR')
class GradReverse(layers.Layer):
    def call(self, x, alpha=1.0): return _grad_reverse(x, tf.cast(alpha, x.dtype))

@keras.utils.register_keras_serializable(package='HAR')
class ChannelIndepTokenizer(layers.Layer):
    '''(B,T,C) -> (B, C*N, D): each channel patchified independently, + channel & positional embeddings.'''
    def __init__(self, patch=PATCH, d_model=D_MODEL, max_channels=32, max_patches=64, **kw):
        super().__init__(**kw); self.patch=patch; self.d_model=d_model
        self.max_channels=max_channels; self.max_patches=max_patches
        self.proj=layers.Dense(d_model)
        self.ch_emb=layers.Embedding(max_channels, d_model)
        self.pos_emb=layers.Embedding(max_patches, d_model)
    def call(self, x):
        B=tf.shape(x)[0]; T=tf.shape(x)[1]; C=tf.shape(x)[2]; N=T//self.patch
        xt=tf.transpose(x,[0,2,1])[:,:,:N*self.patch]
        patches=tf.reshape(xt,[B,C*N,self.patch])
        tok=self.proj(patches)
        ch_idx=tf.repeat(tf.range(C),N); pos_idx=tf.tile(tf.range(N),[C])
        return tok+self.ch_emb(ch_idx)[None]+self.pos_emb(pos_idx)[None], patches, N
    def get_config(self):
        c=super().get_config(); c.update(patch=self.patch,d_model=self.d_model,
            max_channels=self.max_channels,max_patches=self.max_patches); return c

def _tblock(x, d_model, n_heads, ff, drop):
    a=layers.MultiHeadAttention(n_heads, d_model//n_heads, dropout=drop)(x,x)
    x=layers.LayerNormalization(epsilon=1e-6)(x+a)
    g1=layers.Dense(ff,activation='gelu')(x); g2=layers.Dense(ff,activation='sigmoid')(x)
    f=layers.Dense(d_model)(layers.Dropout(drop)(layers.Multiply()([g1,g2])))
    return layers.LayerNormalization(epsilon=1e-6)(x+f)

def build_token_encoder(d_model=D_MODEL, n_layers=N_ENC, name='encoder'):
    inp=keras.Input(shape=(None,d_model)); x=inp
    for _ in range(n_layers): x=_tblock(x,d_model,N_HEADS,FF_DIM,DROPOUT)
    return keras.Model(inp,x,name=name)

class STMAE(keras.Model):
    '''Asymmetric spectro-temporal masked autoencoder (shared across channel counts).'''
    def __init__(self, patch=PATCH, d_model=D_MODEL, mask_ratio=MASK_RATIO,
                 lam_t=LAMBDA_T, lam_f=LAMBDA_F, use_spectral=True, **kw):
        super().__init__(**kw)
        self.tok=ChannelIndepTokenizer(patch,d_model)
        self.encoder=build_token_encoder(d_model,N_ENC,'encoder')
        self.decoder=build_token_encoder(d_model,N_DEC,'decoder')
        self.dec_embed=layers.Dense(d_model)
        self.mask_token=self.add_weight(shape=(1,1,d_model),name='mask_tok',
                                        initializer='zeros',trainable=True)
        self.dec_pos=layers.Embedding(64*32,d_model); self.dec_pred=layers.Dense(patch)
        self.patch=patch; self.d_model=d_model; self.mask_ratio=mask_ratio
        self.lam_t=lam_t; self.lam_f=lam_f; self.use_spectral=use_spectral
        self.loss_tracker=keras.metrics.Mean(name='loss')
    def random_masking(self, tok):
        B=tf.shape(tok)[0]; L=tf.shape(tok)[1]
        keep=tf.cast(tf.cast(L,tf.float32)*(1.0-self.mask_ratio),tf.int32)
        ids=tf.argsort(tf.random.uniform((B,L)),axis=1); restore=tf.argsort(ids,axis=1)
        tok_keep=tf.gather(tok, ids[:,:keep], batch_dims=1)
        mask=tf.gather(tf.concat([tf.zeros((B,keep)),tf.ones((B,L-keep))],1), restore, batch_dims=1)
        return tok_keep, mask, restore, keep
    def call(self, x, training=False):
        tok,patches,N=self.tok(x); B=tf.shape(tok)[0]; L=tf.shape(tok)[1]
        tok_keep,mask,restore,keep=self.random_masking(tok)
        enc=self.dec_embed(self.encoder(tok_keep,training=training))
        mt=tf.tile(self.mask_token,[B,L-keep,1])
        full=tf.gather(tf.concat([enc,mt],1), restore, batch_dims=1)+self.dec_pos(tf.range(L))[None]
        full=self.decoder(full,training=training)
        return self.dec_pred(full), patches, mask
    def _loss(self, pred, target, mask):
        # cast to fp32: rFFT needs fp32, and fp32 reductions are numerically safer under mixed precision
        pred=tf.cast(pred,tf.float32); target=tf.cast(target,tf.float32); mask=tf.cast(mask,tf.float32)
        t=tf.reduce_mean(tf.square(pred-target),-1)
        if self.use_spectral:
            f=tf.reduce_mean(tf.square(tf.abs(tf.signal.rfft(pred))-tf.abs(tf.signal.rfft(target))),-1)
        else:
            f=tf.zeros_like(t)
        per=self.lam_t*t+self.lam_f*f
        return tf.reduce_sum(per*mask)/(tf.reduce_sum(mask)+1e-8)
    def train_step(self, data):
        x=data[0] if isinstance(data,tuple) else data
        lso=hasattr(self.optimizer,'scale_loss')   # Keras-3 mixed-precision LossScaleOptimizer
        with tf.GradientTape() as tape:
            pred,tgt,mask=self(x,training=True); loss=self._loss(pred,tgt,mask)
            obj=self.optimizer.scale_loss(loss) if lso else loss
        g=tape.gradient(obj,self.trainable_variables)
        self.optimizer.apply_gradients(zip(g,self.trainable_variables))  # LSO unscales internally
        self.loss_tracker.update_state(loss); return {'loss':self.loss_tracker.result()}
    @property
    def metrics(self): return [self.loss_tracker]

def build_classifier(tokenizer, encoder, n_classes, n_subjects, d_model=D_MODEL,
                     use_attn_pool=True, use_saft=True):
    inp=keras.Input(shape=(WINDOW_SIZE,None),name='signal')
    tok,_,_=tokenizer(inp); feat=encoder(tok)
    if use_attn_pool:
        aw=layers.Softmax(axis=1)(layers.Dense(1)(feat))
        pooled=layers.Lambda(lambda t: tf.reduce_sum(t[0]*t[1],axis=1))([feat,aw])
    else:
        pooled=layers.GlobalAveragePooling1D()(feat)
    h=layers.Dropout(0.3)(layers.Dense(256,activation='gelu')(pooled))
    y=layers.Dense(n_classes,activation='softmax',name='activity',dtype='float32')(h)
    if use_saft and n_subjects>1:
        rev=GradReverse(name='grl')(pooled)
        s=layers.Dense(128,activation='gelu')(rev)
        s=layers.Dense(n_subjects,activation='softmax',name='subject',dtype='float32')(s)
        return keras.Model(inp,[y,s],name='ParetoHAR_SAFT')
    return keras.Model(inp,y,name='ParetoHAR')
print('Proposed architecture defined (CIT + ST-MAE + SAFT).')


## 8 — Honest baselines (identical LOSO folds, identical metrics)
DeepConvLSTM (Ordóñez & Roggen 2016), a compact CNN-LSTM, a plain Transformer, and the v1 model. All trained supervised on the same splits.

In [ ]:
def build_deepconvlstm(n_ch, n_classes):
    inp=keras.Input((WINDOW_SIZE,n_ch)); x=inp
    for _ in range(4): x=layers.Conv1D(64,5,activation='relu',padding='same')(x)
    x=layers.LSTM(128,return_sequences=True)(x); x=layers.LSTM(128)(x)
    x=layers.Dropout(0.5)(x); out=layers.Dense(n_classes,activation='softmax',dtype='float32')(x)
    return keras.Model(inp,out,name='DeepConvLSTM')

def build_cnn_lstm(n_ch, n_classes):
    inp=keras.Input((WINDOW_SIZE,n_ch))
    x=layers.Conv1D(64,3,activation='relu',padding='same')(inp); x=layers.MaxPool1D(2)(x)
    x=layers.Conv1D(128,3,activation='relu',padding='same')(x); x=layers.MaxPool1D(2)(x)
    x=layers.LSTM(64)(x); x=layers.Dropout(0.4)(x)
    return keras.Model(inp, layers.Dense(n_classes,activation='softmax',dtype='float32')(x), name='CNN-LSTM')

def build_plain_transformer(n_ch, n_classes):
    inp=keras.Input((WINDOW_SIZE,n_ch)); x=layers.Dense(D_MODEL)(inp)
    for _ in range(N_ENC): x=_tblock(x,D_MODEL,N_HEADS,FF_DIM,DROPOUT)
    x=layers.GlobalAveragePooling1D()(x); x=layers.Dropout(0.3)(x)
    return keras.Model(inp, layers.Dense(n_classes,activation='softmax',dtype='float32')(x), name='Transformer')

BASELINE_BUILDERS={'DeepConvLSTM':build_deepconvlstm,'CNN-LSTM':build_cnn_lstm,'Transformer':build_plain_transformer}
print('Baselines:', list(BASELINE_BUILDERS))


## 9 — Evaluation harness: full metric battery + statistical tests + LaTeX

In [ ]:
import warnings
from scipy import stats
from sklearn.metrics import (accuracy_score,f1_score,precision_score,recall_score,
    balanced_accuracy_score,cohen_kappa_score,matthews_corrcoef,confusion_matrix,
    roc_auc_score,classification_report)
from statsmodels.stats.contingency_tables import mcnemar

def expected_calibration_error(y_true, probs, n_bins=15):
    conf=probs.max(1); pred=probs.argmax(1); acc=(pred==y_true).astype(float)
    b=np.linspace(0,1,n_bins+1); ece=0.0
    for i in range(n_bins):
        m=(conf>b[i])&(conf<=b[i+1])
        if m.sum()>0: ece+=m.mean()*abs(acc[m].mean()-conf[m].mean())
    return float(ece)

def safe_macro_auc(y_true, probs, n_classes):
    '''One-vs-rest macro AUC over only the classes present in this fold.
    LOSO-safe: a held-out subject often lacks some activities, which makes the
    standard multiclass AUC undefined (and noisy). We average per-class AUC only
    over present classes that have both positives and negatives.'''
    present=np.unique(y_true)
    if len(present)<2: return float('nan')
    aucs=[]
    for c in present:
        yb=(y_true==c).astype(int)
        if yb.min()==yb.max(): continue       # no negatives or no positives -> undefined, skip
        try: aucs.append(roc_auc_score(yb, probs[:,c]))
        except Exception: pass
    return float(np.mean(aucs)) if aucs else float('nan')

def full_metrics(y_true,y_pred,probs=None,n_classes=None):
    n_classes=n_classes or int(max(y_true.max(),y_pred.max())+1)
    m=dict(accuracy=accuracy_score(y_true,y_pred)*100,
           balanced_acc=balanced_accuracy_score(y_true,y_pred)*100,
           f1_macro=f1_score(y_true,y_pred,average='macro',zero_division=0)*100,
           f1_weighted=f1_score(y_true,y_pred,average='weighted',zero_division=0)*100,
           precision_macro=precision_score(y_true,y_pred,average='macro',zero_division=0)*100,
           recall_macro=recall_score(y_true,y_pred,average='macro',zero_division=0)*100,
           kappa=cohen_kappa_score(y_true,y_pred), mcc=matthews_corrcoef(y_true,y_pred))
    if probs is not None:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            m['auc_macro']=safe_macro_auc(y_true,probs,n_classes)
        m['ece']=expected_calibration_error(y_true,probs)
    return m

def bootstrap_ci(y_true,y_pred,fn,n_boot=1000,seed=SEED):
    r=np.random.default_rng(seed); n=len(y_true); s=[fn(y_true[i],y_pred[i]) for i in (r.integers(0,n,n) for _ in range(n_boot))]
    return float(np.mean(s)),float(np.percentile(s,2.5)),float(np.percentile(s,97.5))

def paired_fold_test(a,b):
    a,b=np.asarray(a),np.asarray(b); d=a-b
    try: w=stats.wilcoxon(a,b).pvalue
    except ValueError: w=float('nan')
    return dict(mean_diff=float(d.mean()), t_pvalue=float(stats.ttest_rel(a,b).pvalue),
                wilcoxon_pvalue=float(w), cohens_d=float(d.mean()/(d.std(ddof=1)+1e-12)))

def mcnemar_test(y_true,pa,pb):
    ac=(pa==y_true); bc=(pb==y_true)
    n01=int(np.sum(ac&~bc)); n10=int(np.sum(~ac&bc))
    res=mcnemar([[0,n01],[n10,0]], exact=(n01+n10<25), correction=True)
    return dict(n01=n01,n10=n10,statistic=float(res.statistic),pvalue=float(res.pvalue))

NEMENYI_Q={2:1.960,3:2.343,4:2.569,5:2.728,6:2.850,7:2.949,8:3.031,9:3.102,10:3.164}
def friedman_nemenyi(scores_by_model):
    names=list(scores_by_model); mat=np.array([scores_by_model[n] for n in names]).T
    fr=stats.friedmanchisquare(*mat.T); k,N=len(names),mat.shape[0]
    ranks=np.array([stats.rankdata(-row) for row in mat]).mean(0)
    cd=NEMENYI_Q.get(k,3.2)*np.sqrt(k*(k+1)/(6.0*N))
    return dict(friedman_stat=float(fr.statistic),friedman_p=float(fr.pvalue),
                avg_ranks=dict(zip(names,ranks.tolist())),cd=float(cd))

def latex_table(rows, caption='Subject-independent (LOSO) results', label='tab:main'):
    cols=['Method','Acc','BalAcc','F1$_{ma}$','F1$_w$','$\\kappa$','MCC']
    s=['\\begin{table}[t]\\centering',f'\\caption{{{caption}}}\\label{{{label}}}',
       '\\begin{tabular}{l'+'c'*(len(cols)-1)+'}\\toprule', ' & '.join(cols)+' \\\\\\midrule']
    for r in rows:
        s.append(f"{r['name']} & {r['accuracy']:.2f} & {r['balanced_acc']:.2f} & "
                 f"{r['f1_macro']:.2f} & {r['f1_weighted']:.2f} & {r['kappa']:.3f} & {r['mcc']:.3f} \\\\")
    return '\n'.join(s+['\\bottomrule\\end{tabular}\\end{table}'])
print('Evaluation harness ready.')


## 10 — Leave-One-Subject-Out training (proposed + baselines, leakage-free)
For each held-out subject: normalise on train only; (proposed) pretrain ST-MAE on **training-subject windows only**; fine-tune with SAFT; evaluate on the held-out subject. Per-fold predictions + metrics are stored for statistics.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# -------------------- training utilities --------------------
def make_optimizer(lr, wd=0.0):
    '''AdamW with gradient clipping; wrapped in a LossScaleOptimizer under mixed precision.'''
    opt = keras.optimizers.AdamW(lr, weight_decay=wd, clipnorm=GRAD_CLIPNORM)
    if mixed_precision.global_policy().name == 'mixed_float16':
        opt = mixed_precision.LossScaleOptimizer(opt)
    return opt

def class_weight_vec(y, n_cls):
    if not USE_CLASS_WEIGHTS: return np.ones(n_cls, np.float32)
    present=np.unique(y)
    w=compute_class_weight('balanced', classes=present, y=y)
    out=np.ones(n_cls, np.float32)
    for c,wi in zip(present, w): out[c]=wi
    return out.astype(np.float32)

def fold_normalise(Xtr, *others):
    sc=StandardScaler().fit(Xtr.reshape(-1,Xtr.shape[-1]))
    def ap(A): return sc.transform(A.reshape(-1,A.shape[-1])).reshape(A.shape).astype(np.float32)
    return (ap(Xtr),)+tuple(ap(o) for o in others)

def saft_alpha(epoch, total):
    p=epoch/max(1,total); return SAFT_MAX_ALPHA*(2.0/(1.0+np.exp(-SAFT_GAMMA*p))-1.0)

class AlphaRamp(keras.callbacks.Callback):
    '''Sets the SAFT adversary strength each epoch WITHOUT retracing (alpha is a tf.Variable).'''
    def __init__(self, trainer, total, use_saft):
        self.t=trainer; self.total=total; self.use_saft=use_saft
    def on_epoch_begin(self, epoch, logs=None):
        self.t.alpha.assign(saft_alpha(epoch,self.total) if self.use_saft else 0.0)

class SAFTTrainer(keras.Model):
    '''Classifier + subject-adversary; class-weighted activity loss, GRL-driven invariance,
       mixed-precision loss scaling, and val-based model selection.'''
    def __init__(self, model, n_subjects, class_w=None, **kw):
        super().__init__(**kw); self.model=model; self.n_subjects=n_subjects
        self.alpha=tf.Variable(0.0,trainable=False,dtype=tf.float32)
        self.dual=isinstance(model.output,list)
        self.class_w=tf.constant(class_w if class_w is not None else
                                 np.ones(model.output[0].shape[-1] if self.dual else model.output.shape[-1],
                                         np.float32))
        self.am=keras.metrics.Mean('loss'); self.acc=keras.metrics.SparseCategoricalAccuracy('acc')
    def compile(self, opt):
        super().compile(jit_compile=(USE_XLA and len(GPUS)>0), steps_per_execution=STEPS_PER_EXEC)
        self.optimizer=opt
    def _act_loss(self, ya, pa):
        ce=keras.losses.sparse_categorical_crossentropy(ya,pa)
        return tf.reduce_mean(ce*tf.gather(self.class_w, ya))
    def train_step(self, data):
        x,(ya,ys)=data
        lso=hasattr(self.optimizer,'scale_loss')   # Keras-3 mixed-precision LossScaleOptimizer
        with tf.GradientTape() as tape:
            if self.dual:
                pa,ps=self.model(x,training=True)
                ld=tf.reduce_mean(keras.losses.sparse_categorical_crossentropy(ys,ps))
                loss=self._act_loss(ya,pa)+self.alpha*ld
            else:
                pa=self.model(x,training=True); loss=self._act_loss(ya,pa)
            obj=self.optimizer.scale_loss(loss) if lso else loss
        g=tape.gradient(obj,self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(g,self.model.trainable_variables))  # LSO unscales internally
        self.am.update_state(loss); self.acc.update_state(ya,pa)
        return {'loss':self.am.result(),'acc':self.acc.result()}
    def test_step(self, data):
        x,(ya,ys)=data
        pa=self.model(x,training=False); pa=pa[0] if isinstance(pa,list) else pa
        self.am.update_state(self._act_loss(ya,pa)); self.acc.update_state(ya,pa)
        return {'loss':self.am.result(),'acc':self.acc.result()}
    @property
    def metrics(self): return [self.am,self.acc]

def _ds(X, y, batch, training, second):
    '''Static-shape tf.data pipeline: (X,(y,second)). drop_remainder on train for XLA/tensor cores.'''
    b=min(batch, len(X)) if training else batch          # never let drop_remainder empty a small set
    d=tf.data.Dataset.from_tensor_slices((X,(y,second)))
    if training: d=d.shuffle(min(8192,len(X)),seed=SEED,reshuffle_each_iteration=True)
    d=d.batch(b, drop_remainder=training).prefetch(tf.data.AUTOTUNE)
    return d

def pretrain_stmae(X_pool, use_spectral=True, epochs=MAE_EPOCHS):
    mae=STMAE(use_spectral=use_spectral)
    mae.compile(optimizer=make_optimizer(MAE_LR, wd=1e-4),
                jit_compile=(USE_XLA and len(GPUS)>0), steps_per_execution=STEPS_PER_EXEC)
    ds=tf.data.Dataset.from_tensor_slices(X_pool).shuffle(min(8192,len(X_pool)),seed=SEED)\
        .batch(min(MAE_BATCH,len(X_pool)), drop_remainder=True).prefetch(tf.data.AUTOTUNE)
    mae.fit(ds, epochs=epochs, verbose=0,
            callbacks=[keras.callbacks.EarlyStopping('loss',patience=6,restore_best_weights=True),
                       keras.callbacks.TerminateOnNaN()])
    return mae

# -------------------- resume cache + backbone checkpoints --------------------
def fold_key(ds, tag, s): return f"{ds}__{tag}__fold{s}".replace('/','_')
def _cache_path(key): return f"{FOLDCACHE_DIR}/{key}.npz"
def save_fold_cache(key, m, y_true, y_pred):
    np.savez_compressed(_cache_path(key), y_true=y_true, y_pred=y_pred, metrics=json.dumps(m))
def load_fold_cache(key):
    p=_cache_path(key)
    if not os.path.exists(p): return None
    d=np.load(p, allow_pickle=True)
    return json.loads(str(d['metrics'])), d['y_true'], d['y_pred']
def save_backbone(tokenizer, encoder, key):
    if not SAVE_CHECKPOINTS: return
    inp=keras.Input((WINDOW_SIZE,None)); tok,_,_=tokenizer(inp)
    keras.Model(inp, encoder(tok)).save_weights(f"{CKPT_DIR}/backbone_{key}.weights.h5")

def run_loso(dsname, builder=None, proposed=False, use_saft=True, use_spectral=True,
             use_attn_pool=True, pretrain=True, max_folds=LOSO_MAX_FOLDS, tag=None):
    X,y,g,le=DATA[dsname]; n_cls=len(le.classes_); n_ch=X.shape[-1]; subs=np.unique(g)
    if max_folds: subs=subs[:max_folds]
    tag=tag or ('Proposed' if proposed else builder.__name__)
    fold_metrics=[]; pooled_true=[]; pooled_pred=[]
    ep=(8 if QUICK_SMOKE else FT_EPOCHS); pe=(4 if QUICK_SMOKE else MAE_EPOCHS)
    for fi,s in enumerate(subs):
        key=fold_key(dsname,tag,s)
        # ---- RESUME: skip folds already computed in a previous (possibly interrupted) run ----
        if RESUME:
            cached=load_fold_cache(key)
            if cached is not None:
                m,yt,yp=cached; fold_metrics.append(m)
                pooled_true.append(yt); pooled_pred.append(yp)
                print(f'  [resume {tag}|{dsname}] subj {s}: acc={m["accuracy"]:.2f}'); continue
        te=(g==s); trv=~te
        Xtr,ytr,gtr=X[trv],y[trv],g[trv]; Xte,yte=X[te],y[te]
        if len(np.unique(yte))<2 or len(Xte)<10: continue
        Xtr,Xte=fold_normalise(Xtr,Xte)                              # leakage-free
        su=np.unique(gtr); s2i={u:i for i,u in enumerate(su)}; str_=np.array([s2i[u] for u in gtr])
        Xtr2,Xv,ytr2,yv,str2,_=train_test_split(Xtr,ytr,str_,test_size=0.15,random_state=SEED,
                                                 stratify=ytr if min(np.bincount(ytr))>=2 else None)
        cw=class_weight_vec(ytr2, n_cls)
        cb=[keras.callbacks.EarlyStopping('val_acc',patience=12,mode='max',restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau('val_loss',factor=0.5,patience=6,min_lr=1e-6),
            keras.callbacks.TerminateOnNaN()]
        if proposed:
            if pretrain:
                mae=pretrain_stmae(Xtr2, use_spectral=use_spectral, epochs=pe); tok,enc=mae.tok,mae.encoder
            else:
                tmp=STMAE(use_spectral=use_spectral); _=tmp(Xtr2[:2]); tok,enc=tmp.tok,tmp.encoder
            clf=build_classifier(tok,enc,n_cls,len(su),use_attn_pool=use_attn_pool,use_saft=use_saft)
            trainer=SAFTTrainer(clf,len(su),class_w=cw); trainer.compile(make_optimizer(FT_LR, wd=1e-5))
            tr_ds=_ds(Xtr2,ytr2,FT_BATCH,True,second=str2)
            zv=np.zeros_like(yv)                          # dummy subject ids for val (unused in test_step)
            va_ds=_ds(Xv,yv,FT_BATCH,False,second=zv)
            trainer.fit(tr_ds, validation_data=va_ds, epochs=ep, verbose=0,
                        callbacks=cb+[AlphaRamp(trainer,ep,use_saft)])   # single fit -> no per-epoch retrace
            probs=clf.predict(Xte,batch_size=512,verbose=0)
            probs=probs[0] if isinstance(probs,list) else probs
            save_backbone(tok,enc,key)
        else:
            model=builder(n_ch,n_cls)
            model.compile(make_optimizer(FT_LR, wd=1e-5),'sparse_categorical_crossentropy',
                          metrics=[keras.metrics.SparseCategoricalAccuracy('acc')],
                          jit_compile=(USE_XLA and len(GPUS)>0), steps_per_execution=STEPS_PER_EXEC)
            cwd={i:float(cw[i]) for i in range(n_cls)}
            model.fit(Xtr2, ytr2, validation_data=(Xv,yv), epochs=ep, batch_size=FT_BATCH, verbose=0,
                      class_weight=(cwd if USE_CLASS_WEIGHTS else None), callbacks=cb)
            probs=model.predict(Xte,batch_size=512,verbose=0)
        pred=probs.argmax(1); m=full_metrics(yte,pred,probs,n_cls); m['fold']=int(s)
        fold_metrics.append(m); pooled_true.append(yte); pooled_pred.append(pred)
        save_fold_cache(key, m, yte, pred)                # ---- CHECKPOINT this fold for resume ----
        journal.log('loso',dsname,int(s),ep,{'model':tag,**{k:v for k,v in m.items() if k!='fold'}})
        print(f'  [{tag}|{dsname}] fold {fi+1}/{len(subs)} subj {s}: acc={m["accuracy"]:.2f} f1m={m["f1_macro"]:.2f}')
    return dict(tag=tag,dataset=dsname,folds=fold_metrics,
                y_true=np.concatenate(pooled_true),y_pred=np.concatenate(pooled_pred))
print('LOSO harness ready (mixed precision / XLA / resume / checkpoints).')


In [ ]:
# ---- RUN: proposed + baselines across datasets (this is the long cell) ----
RESULTS={}   # RESULTS[dataset][model_tag] = run dict
for ds in DATASETS_TO_RUN:
    RESULTS.setdefault(ds,{})
    print(f'\n===== {ds} :: PROPOSED =====')
    RESULTS[ds]['Proposed']=run_loso(ds, proposed=True, use_saft=(SAFT_MAX_ALPHA>0),
                                     use_spectral=True, pretrain=True)
    if RUN_BASELINES:
        for bname,bfn in BASELINE_BUILDERS.items():
            print(f'===== {ds} :: {bname} =====')
            RESULTS[ds][bname]=run_loso(ds, builder=bfn, proposed=False)
print('\nAll LOSO runs complete.')


## 11 — Ablation study (proposed components turned off one at a time)
`−SAFT` (no adversary), `−Spectral` (time-only recon), `−Pretrain` (random init), `−AttnPool` (mean pooling). Headline dataset only by default to save compute.

In [ ]:
ABLATION={}
if RUN_ABLATIONS:
    ads=DATASETS_TO_RUN[0]
    print(f'Ablations on {ads}')
    ABLATION['Full']           = RESULTS[ads]['Proposed']
    ABLATION['-SAFT']          = run_loso(ads, proposed=True, use_saft=False, use_spectral=True,  pretrain=True,  tag='-SAFT')
    ABLATION['-Spectral']      = run_loso(ads, proposed=True, use_saft=True,  use_spectral=False, pretrain=True,  tag='-Spectral')
    ABLATION['-Pretrain']      = run_loso(ads, proposed=True, use_saft=True,  use_spectral=True,  pretrain=False, tag='-Pretrain')
    ABLATION['-AttnPool']      = run_loso(ads, proposed=True, use_saft=True,  use_spectral=True,  pretrain=True,  use_attn_pool=False, tag='-AttnPool')
    print('Ablation complete.')


## 12 — Statistical significance & summary tables
Per-fold means ± std, bootstrap CIs, paired Wilcoxon/t-test + Cohen's d vs each baseline, McNemar on pooled predictions, Friedman + Nemenyi across models.

In [ ]:
def fold_scores(run, key='accuracy'): return [f[key] for f in run['folds']]
def summarise(run):
    fm=run['folds']; agg={}
    for k in fm[0]:
        if k=='fold': continue
        v=np.array([f[k] for f in fm]); agg[k]=float(np.nanmean(v)); agg[k+'_std']=float(np.nanstd(v))
    return agg

SUMMARY_ROWS=[]
for ds in DATASETS_TO_RUN:
    print(f'\n################ {ds} ################')
    rows=[]
    for tag,run in RESULTS[ds].items():
        s=summarise(run); rows.append(dict(name=tag,**s))
        mean,lo,hi=bootstrap_ci(run['y_true'],run['y_pred'],lambda a,b:accuracy_score(a,b)*100,500)
        print(f"{tag:14s} acc={s['accuracy']:.2f}±{s['accuracy_std']:.2f}  "
              f"f1m={s['f1_macro']:.2f}  kappa={s['kappa']:.3f}  MCC={s['mcc']:.3f}  "
              f"boot95%CI[{lo:.2f},{hi:.2f}]")
        s['dataset']=ds; SUMMARY_ROWS.append(s | {'name':tag})
    # significance vs proposed
    prop=RESULTS[ds]['Proposed']
    print('  --- Proposed vs baselines ---')
    for tag,run in RESULTS[ds].items():
        if tag=='Proposed': continue
        pt=paired_fold_test(fold_scores(prop),fold_scores(run))
        mc=mcnemar_test(prop['y_true'],prop['y_pred'],
                        run['y_pred'] if len(run['y_pred'])==len(prop['y_pred']) else prop['y_pred'])
        print(f"   vs {tag:12s} Δacc={pt['mean_diff']:+.2f}  Wilcoxon p={pt['wilcoxon_pvalue']:.4f}  "
              f"t p={pt['t_pvalue']:.4f}  d={pt['cohens_d']:.2f}  McNemar p={mc['pvalue']:.2e}")
    # Friedman + Nemenyi (need >=2 baselines)
    sbm={tag:fold_scores(run) for tag,run in RESULTS[ds].items()}
    lens={len(v) for v in sbm.values()}
    if len(sbm)>=3 and len(lens)==1:
        fn=friedman_nemenyi(sbm)
        print(f"  Friedman chi2={fn['friedman_stat']:.2f} p={fn['friedman_p']:.4f}  CD={fn['cd']:.3f}")
        print('  ranks:', {k:round(v,2) for k,v in fn['avg_ranks'].items()})

# LaTeX for headline dataset
print('\n===== LaTeX (headline dataset) =====')
hd=DATASETS_TO_RUN[0]
print(latex_table([dict(name=t,**summarise(r)) for t,r in RESULTS[hd].items()],
                  caption=f'Subject-independent LOSO results on {hd}'))


## 13 — Pareto & efficiency (accuracy vs params / latency / FLOPs)
The "Pareto" in Pareto-HAR: a method is only interesting if it improves accuracy *without* exploding compute.

In [ ]:
def measure_efficiency(model, n_ch):
    xb=tf.random.normal((1,WINDOW_SIZE,n_ch))
    _=model(xb)  # build
    import time
    # latency (ms/window, batched)
    big=tf.random.normal((256,WINDOW_SIZE,n_ch)); _=model(big)
    t0=time.time()
    for _ in range(5): _=model(big)
    lat=(time.time()-t0)/5/256*1000
    params=model.count_params()
    flops=float('nan')
    try:
        from tensorflow.python.profiler.model_analyzer import profile
        from tensorflow.python.profiler.option_builder import ProfileOptionBuilder
        cf=tf.function(lambda x: model(x)).get_concrete_function(tf.TensorSpec([1,WINDOW_SIZE,n_ch]))
        g=cf.graph; opts=ProfileOptionBuilder.float_operation(); opts['output']='none'
        flops=profile(g,options=opts).total_float_ops
    except Exception: pass
    size_mb=params*4/1e6
    return dict(params=int(params), flops=flops, latency_ms=float(lat), size_mb=float(size_mb))

EFF={}
hd=DATASETS_TO_RUN[0]; n_ch=DATA[hd][0].shape[-1]; n_cls=len(DATA[hd][3].classes_)
tmp=STMAE(); _=tmp(tf.random.normal((2,WINDOW_SIZE,n_ch)))
EFF['Proposed']=measure_efficiency(build_classifier(tmp.tok,tmp.encoder,n_cls,5,use_saft=False), n_ch)
for bname,bfn in BASELINE_BUILDERS.items():
    EFF[bname]=measure_efficiency(bfn(n_ch,n_cls), n_ch)
for k,v in EFF.items():
    print(f"{k:14s} params={v['params']:>10,}  latency={v['latency_ms']:.3f} ms/win  size={v['size_mb']:.2f} MB")


## 14 — Publication visualisations
Confusion matrices, training curves, **t-SNE coloured by activity and by subject** (visual evidence of subject-invariance), reliability diagram, Pareto front, and the critical-difference diagram.

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
sns.set_context('paper'); plt.rcParams['figure.dpi']=120

# (a) Confusion matrix — proposed, headline dataset
def plot_cm(run, le, fname):
    cm=confusion_matrix(run['y_true'],run['y_pred'])
    cmn=cm/cm.sum(1,keepdims=True).clip(min=1)
    fig,ax=plt.subplots(figsize=(7,6))
    sns.heatmap(cmn,annot=True,fmt='.2f',cmap='viridis',
                xticklabels=le.classes_,yticklabels=le.classes_,ax=ax,cbar_kws={'label':'recall'})
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(f'{run["tag"]} — {run["dataset"]} (norm.)')
    plt.xticks(rotation=45,ha='right'); plt.tight_layout(); plt.savefig(fname,bbox_inches='tight'); plt.show()
plot_cm(RESULTS[hd]['Proposed'], DATA[hd][3], f'{PLOTS_DIR}/cm_proposed_{hd}.png')

# (b) Pareto front: accuracy vs params
fig,ax=plt.subplots(figsize=(6,5))
for tag,run in RESULTS[hd].items():
    acc=np.mean([f['accuracy'] for f in run['folds']])
    p=EFF.get(tag,{}).get('params',np.nan)
    ax.scatter(p,acc,s=90); ax.annotate(tag,(p,acc),textcoords='offset points',xytext=(6,4))
ax.set_xscale('log'); ax.set_xlabel('Parameters (log)'); ax.set_ylabel('LOSO accuracy (%)')
ax.set_title(f'Accuracy–compute Pareto ({hd})'); plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/pareto_{hd}.png',bbox_inches='tight'); plt.show()

# (c) Reliability diagram (calibration) — proposed
def reliability(y_true,probs,fname,n_bins=12):
    conf=probs.max(1); pred=probs.argmax(1); acc=(pred==y_true).astype(float)
    b=np.linspace(0,1,n_bins+1); xs=[]; ys=[]
    for i in range(n_bins):
        m=(conf>b[i])&(conf<=b[i+1])
        if m.sum()>0: xs.append(conf[m].mean()); ys.append(acc[m].mean())
    fig,ax=plt.subplots(figsize=(5,5)); ax.plot([0,1],[0,1],'--',c='gray')
    ax.plot(xs,ys,'o-'); ax.set_xlabel('confidence'); ax.set_ylabel('accuracy')
    ax.set_title('Reliability diagram'); plt.tight_layout(); plt.savefig(fname,bbox_inches='tight'); plt.show()
# need probs on pooled test — recompute quickly skipped; use journal if available

# (d) Critical-difference diagram (Nemenyi)
def cd_diagram(scores_by_model, fname):
    names=list(scores_by_model); mat=np.array([scores_by_model[n] for n in names]).T
    if len({len(v) for v in scores_by_model.values()})!=1 or len(names)<3: 
        print('CD diagram needs >=3 models, equal folds'); return
    fn=friedman_nemenyi(scores_by_model); ranks=fn['avg_ranks']; cd=fn['cd']
    order=sorted(ranks,key=ranks.get); k=len(names)
    fig,ax=plt.subplots(figsize=(8,2.2)); ax.set_xlim(1,k); ax.set_ylim(0,1); ax.axis('off')
    ax.plot([1,k],[0.8,0.8],'k'); 
    for r in range(1,k+1): ax.plot([r,r],[0.78,0.82],'k'); ax.text(r,0.86,str(r),ha='center')
    for i,name in enumerate(order):
        y=0.6-0.12*(i%4); ax.plot([ranks[name],ranks[name]],[0.78,y],'k',lw=0.7)
        ax.text(ranks[name],y-0.03,f'{name} ({ranks[name]:.2f})',ha='center',va='top',fontsize=8)
    ax.plot([1,1+cd],[0.95,0.95],'r',lw=2); ax.text(1+cd/2,0.97,f'CD={cd:.2f}',ha='center',color='r',fontsize=8)
    plt.title('Critical-difference diagram (Nemenyi, α=0.05)'); plt.tight_layout()
    plt.savefig(fname,bbox_inches='tight'); plt.show()
sbm={t:[f['accuracy'] for f in r['folds']] for t,r in RESULTS[hd].items()}
cd_diagram(sbm, f'{PLOTS_DIR}/cd_{hd}.png')

# (e) Ablation bar chart
if RUN_ABLATIONS and ABLATION:
    fig,ax=plt.subplots(figsize=(6,4))
    names=list(ABLATION); accs=[np.mean([f['accuracy'] for f in ABLATION[n]['folds']]) for n in names]
    f1s=[np.mean([f['f1_macro'] for f in ABLATION[n]['folds']]) for n in names]
    x=np.arange(len(names)); w=0.38
    ax.bar(x-w/2,accs,w,label='Accuracy'); ax.bar(x+w/2,f1s,w,label='F1-macro')
    ax.set_xticks(x); ax.set_xticklabels(names,rotation=20); ax.set_ylabel('%'); ax.legend()
    ax.set_title('Ablation'); plt.tight_layout(); plt.savefig(f'{PLOTS_DIR}/ablation.png',bbox_inches='tight'); plt.show()


In [ ]:
# (f) t-SNE of embeddings, coloured by activity AND by subject (subject-invariance evidence)
from sklearn.manifold import TSNE
def embed_and_tsne(dsname):
    X,y,g,le=DATA[dsname]; subs=np.unique(g)[:LOSO_MAX_FOLDS or 5]
    m=np.isin(g,subs); Xe,ye,ge=X[m],y[m],g[m]
    Xe=StandardScaler().fit_transform(Xe.reshape(-1,Xe.shape[-1])).reshape(Xe.shape).astype(np.float32)
    idx=np.random.default_rng(0).choice(len(Xe),min(1500,len(Xe)),replace=False)
    mae=STMAE(); _=mae(Xe[:2])
    enc=keras.Model(mae.tok.input if hasattr(mae.tok,'input') else None, None) if False else None
    # build feature extractor: tokenizer->encoder->mean
    inp=keras.Input((WINDOW_SIZE,Xe.shape[-1])); tok,_,_=mae.tok(inp); f=mae.encoder(tok)
    fe=keras.Model(inp, layers.GlobalAveragePooling1D()(f))
    Z=fe.predict(Xe[idx],batch_size=256,verbose=0)
    Z2=TSNE(n_components=2,init='pca',perplexity=30,random_state=0).fit_transform(Z)
    fig,axs=plt.subplots(1,2,figsize=(12,5))
    sc0=axs[0].scatter(Z2[:,0],Z2[:,1],c=ye[idx],cmap='tab10',s=8); axs[0].set_title('coloured by activity')
    sc1=axs[1].scatter(Z2[:,0],Z2[:,1],c=pd.factorize(ge[idx])[0],cmap='tab20',s=8); axs[1].set_title('coloured by subject\n(good = subjects mixed)')
    for a in axs: a.set_xticks([]); a.set_yticks([])
    plt.suptitle(f't-SNE of encoder features ({dsname})'); plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/tsne_{dsname}.png',bbox_inches='tight'); plt.show()
embed_and_tsne(hd)
print('Note: this t-SNE uses a *random-init* encoder for a quick look. For the paper figure,\n'
      'extract features from a trained fold model instead (swap mae.encoder for the fine-tuned encoder).')


## 15 — Export everything (CSV, JSON, LaTeX, plots) to the persistent results folder

In [ ]:
import json
# tidy per-fold table
recs=[]
for ds in DATASETS_TO_RUN:
    for tag,run in RESULTS[ds].items():
        for f in run['folds']:
            recs.append({'dataset':ds,'model':tag,**f})
fold_df=pd.DataFrame(recs); fold_df.to_csv(f'{RESULTS_DIR}/loso_per_fold_{RUN_ID}.csv',index=False)

summary_df=pd.DataFrame(SUMMARY_ROWS)
summary_df.to_csv(f'{RESULTS_DIR}/loso_summary_{RUN_ID}.csv',index=False)

# LaTeX tables for every dataset
with open(f'{RESULTS_DIR}/tables_{RUN_ID}.tex','w') as f:
    for ds in DATASETS_TO_RUN:
        f.write(latex_table([dict(name=t,**summarise(r)) for t,r in RESULTS[ds].items()],
                            caption=f'LOSO results on {ds}', label=f'tab:{ds.lower()}')+'\n\n')

# efficiency table
pd.DataFrame(EFF).T.to_csv(f'{RESULTS_DIR}/efficiency_{RUN_ID}.csv')

print('Saved to', RESULTS_DIR)
print(' - loso_per_fold_*.csv  (every fold, every metric)')
print(' - loso_summary_*.csv   (mean±std per model/dataset)')
print(' - tables_*.tex         (paste into your manuscript)')
print(' - efficiency_*.csv     (params/latency)')
print(' - plots/*.png          (CM, Pareto, CD, ablation, t-SNE)')
fold_df.head()


## 16 — Resume inspector & progress

The study **resumes automatically**: every completed `(dataset, model, fold)` is cached to
`results/foldcache/`, and re-running the long training cell after a Studio restart or GPU swap skips
everything already done. This cell shows what is finished and what remains.

In [ ]:
import glob
done=sorted(os.path.basename(p)[:-4] for p in glob.glob(f'{FOLDCACHE_DIR}/*.npz'))
print(f'Completed folds cached: {len(done)}')
if done:
    prog=pd.DataFrame([d.split('__') for d in done], columns=['dataset','model','fold'])
    print(prog.groupby(['dataset','model']).size().rename('folds_done'))
j=journal.df()
if len(j):
    print('\nJournal entries:', len(j))
    print(j[j.phase=='loso'].groupby(['dataset','model'])['accuracy'].agg(['mean','std','count']))

# --- controls ---
# Re-run a single fold: delete its cache file, e.g.
#   os.remove(f'{FOLDCACHE_DIR}/WISDM__Proposed__fold1.npz')
# Fresh slate for everything: set a new RUN_ID, or clear the cache dir:
#   import shutil; shutil.rmtree(FOLDCACHE_DIR); os.makedirs(FOLDCACHE_DIR, exist_ok=True)


---
### How to use this notebook (Lightning AI)
1. Open the notebook in a **Lightning AI Studio** and attach a GPU (any CUDA GPU works; bigger VRAM → larger batches).
2. **Validate fast:** set `QUICK_SMOKE=True`, `LOSO_MAX_FOLDS=2` → full pipeline in minutes.
3. **Real run:** `QUICK_SMOKE=False`, `LOSO_MAX_FOLDS=5` (or `None` for all subjects).
4. **Resume after a stop / GPU swap:** just re-run the cells top-to-bottom. The Studio filesystem is persistent,
   so with `RESUME=True` and the fixed `RUN_ID='main'`, completed folds load from cache and only unfinished work trains.
5. **Use the GPU hard:** on an A100 the notebook automatically uses **bfloat16** tensor cores (fp16 on older cards),
   **XLA**-compiled train steps, **steps_per_execution=32** (fuses 32 train steps per GPU dispatch — the biggest
   utilisation win for small models), and **auto-scales batches** to your VRAM (×2 at 40GB, ×4 at 80GB, with
   sqrt-scaled LRs). If you OOM, set `AUTO_SCALE_BATCH=False` or lower `MAE_BATCH`/`FT_BATCH`; if XLA errors
   on your shapes, set `USE_XLA=False`.
6. **Outputs** land in `OUTPUT_ROOT` (set in cell 3) — CSVs, LaTeX tables, plots, the metrics journal, the resume
   cache, and per-fold backbone checkpoints. Override `OUTPUT_ROOT` (or the `PARETO_ROOT` env var) to relocate them.
7. Toggle `SAFT_MAX_ALPHA=0` for the no-adversary variant; the ablation cell does this automatically.

See `METHODS.md` for the formal problem statement, all equations, the novelty argument, and the SOTA comparison table.